<a href="https://colab.research.google.com/github/chiptune234u-lgtm/SandProject/blob/main/test_sand.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

#@title ⏳ サンドボックス・エディター（30s解放版）

#@markdown ### 1. サイクル設定
CYCLE_DURATION = "20s" #@param ["20s", "30s"]

# ここでDURATIONを確定
if CYCLE_DURATION == "30s":
    DURATION = 30
else:
    DURATION = 20

#@markdown ### 2. 基本設定
MODE = "sand"
FPS = 25
BLOCK_SIZE = 12

#@markdown ### 3. 演出設定
PREFILL_HEIGHT = 0.4

#@markdown ### 4. 風設定
WIND_STRENGTH = 0.3

import numpy as np
import cv2
import os
import glob
import random
from google.colab import drive
import colorsys

# ドライブマウント
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

WIDTH, HEIGHT = 720, 1280
COLS, ROWS = WIDTH // BLOCK_SIZE, HEIGHT // BLOCK_SIZE

# --- 動的な色生成ロジック ---
BASE_HUE = random.random()

def get_dynamic_color():
    h = (BASE_HUE + random.uniform(-0.05, 0.05)) % 1.0
    s = random.uniform(0.5, 0.8)
    v = random.uniform(0.8, 1.0)
    r, g, b = colorsys.hsv_to_rgb(h, s, v)
    return [int(r*255), int(g*255), int(b*255)]

# 背景準備
bg_dir = "/content/drive/MyDrive/Colab Notebooks/画像素材"
bg_files = glob.glob(os.path.join(bg_dir, "*"))
if bg_files:
    bg_img = cv2.imread(random.choice(bg_files))
    bg_img = cv2.resize(bg_img, (WIDTH, HEIGHT))
    bg_img = cv2.cvtColor(bg_img, cv2.COLOR_BGR2RGB)
else:
    bg_img = np.zeros((HEIGHT, WIDTH, 3), dtype=np.uint8)

# --- グリッド初期化（天井に砂をセット） ---
grid = np.zeros((ROWS, COLS, 3), dtype=np.uint8)
fill_limit = int(ROWS * PREFILL_HEIGHT)

for y in range(0, fill_limit):
    width_at_y = int((fill_limit - y) * 1.5)
    center = COLS // 2
    for x in range(center - width_at_y, center + width_at_y):
        if 0 <= x < COLS and np.random.random() > 0.1:
            grid[y, x] = get_dynamic_color()

print(f"初期化完了: モード={DURATION}s, 天井砂セット完了！")

初期化完了: モード=20s, 天井砂セット完了！


In [ ]:
velocity_grid = np.zeros((ROWS, COLS, 2), dtype=np.float32)

def apply_deep_explosion(current_grid, v_grid, power=30):
    active_y, active_x = np.where(np.any(current_grid > 0, axis=2))
    if len(active_y) == 0:
        return v_grid, (COLS//2, ROWS//2)
    min_x, max_x = np.min(active_x), np.max(active_x)
    explosion_x = random.randint(min_x, max_x)
    min_y, max_y = np.min(active_y), np.max(active_y)
    explosion_y = int(max_y - (max_y - min_y) * 0.2)
    for y in range(ROWS):
        for x in range(COLS):
            if np.any(current_grid[y, x]):
                dx, dy = x - explosion_x, y - explosion_y
                dist = np.sqrt(dx**2 + dy**2) + 0.1
                if dist < 60:
                    force = power / dist
                    v_grid[y, x, 0] += (dx / dist) * force * 15
                    v_grid[y, x, 1] += (dy / dist) * force * 15
    return v_grid, (explosion_x, explosion_y)

def apply_wind(current_grid, v_grid, wind_force):
    sand_mask = np.any(current_grid > 0, axis=2)
    sand_mask[ROWS-1, :] = False
    v_grid[sand_mask, 0] += wind_force
    return v_grid

def update_physics_vectorized(current_grid, v_grid, mode):
    new_grid = np.zeros_like(current_grid)
    new_v_grid = np.zeros_like(v_grid)
    for y in range(ROWS - 1, -1, -1):
        sand_mask_in_row = np.any(current_grid[y, :, :], axis=1)
        sand_x_coords = np.where(sand_mask_in_row)[0]
        if sand_x_coords.size == 0: continue
        colors = current_grid[y, sand_x_coords]
        velocities = v_grid[y, sand_x_coords]
        vx, vy = velocities[:, 0], velocities[:, 1]
        proposed_nx = np.clip((sand_x_coords + vx).astype(int), 0, COLS - 1)
        proposed_ny = (y + vy + 1).astype(int)
        new_vx, new_vy = vx * 0.9, vy * 0.9

        # Bottom boundary
        falling_off_mask = proposed_ny >= ROWS
        if np.any(falling_off_mask):
            target_x_fall = proposed_nx[falling_off_mask]
            new_grid[ROWS-1, target_x_fall] = colors[falling_off_mask]
            new_v_grid[ROWS-1, target_x_fall] = 0.0

        remaining_mask = ~falling_off_mask
        if not np.any(remaining_mask): continue
        sand_x_coords_rem = sand_x_coords[remaining_mask]
        colors_rem, proposed_nx_rem, proposed_ny_rem = colors[remaining_mask], proposed_nx[remaining_mask], proposed_ny[remaining_mask]
        new_vx_rem, new_vy_rem = new_vx[remaining_mask], new_vy[remaining_mask]
        proposed_ny_rem = np.clip(proposed_ny_rem, 0, ROWS - 1)

        # Direct move
        target_coords = np.stack([proposed_ny_rem, proposed_nx_rem], axis=1)
        unique_target_coords, unique_target_indices = np.unique(target_coords, axis=0, return_index=True)
        can_move_to_empty = np.array([not np.any(new_grid[ty, tx]) for ty, tx in unique_target_coords])
        direct_indices = unique_target_indices[can_move_to_empty]
        if direct_indices.size > 0:
            new_grid[proposed_ny_rem[direct_indices], proposed_nx_rem[direct_indices]] = colors_rem[direct_indices]
            new_v_grid[proposed_ny_rem[direct_indices], proposed_nx_rem[direct_indices]] = np.stack([new_vx_rem[direct_indices], new_vy_rem[direct_indices]], axis=1)

        # Stuck particles (Sliding)
        stuck_mask = np.ones(len(sand_x_coords_rem), dtype=bool)
        stuck_mask[direct_indices] = False
        stuck_indices = np.where(stuck_mask)[0]
        if stuck_indices.size > 0:
            # --- 左右の優先順位をランダム化して偏りを防ぐ ---
            side_choices = [-1, 1]
            random.shuffle(side_choices)
            side_range = [0] + side_choices if mode == "sand" else [0]

            for i in stuck_indices:
                curr_x, curr_c, curr_vx = sand_x_coords_rem[i], colors_rem[i], new_vx_rem[i]
                moved = False
                for dx in side_range:
                    tx, ty = curr_x + dx, y + 1
                    if 0 <= tx < COLS and ty < ROWS and not np.any(new_grid[ty, tx]):
                        new_grid[ty, tx], new_v_grid[ty, tx] = curr_c, [curr_vx * 0.5, 0]
                        moved = True; break
                if not moved:
                    new_grid[y, curr_x], new_v_grid[y, curr_x] = curr_c, [0, 0]
    return new_grid, new_v_grid

In [ ]:

import time
import math

# 第1セルのDURATIONを元に全フレーム数を確定
total_frames = FPS * DURATION
frames_out = []

# --- タイミングの比率計算 ---
ratio = DURATION / 20.0
EXPLOSION_FRAMES = [0, int(FPS * 15 * ratio)]
EMIT_STOP_F = int(FPS * 17 * ratio)

# 回転演出
ROT_SEC = 2.5
ROTATION_START_F = total_frames - int(FPS * ROT_SEC)

# 風のパラメータ（固定）
WIND_MAX_STRENGTH = WIND_STRENGTH
WIND_START_F = FPS * 7
WIND_PEAK_F  = FPS * 10
WIND_END_F   = FPS * 13
WIND_DIRECTION = random.choice([-1, 1])

print(f"🚀 {DURATION}秒ループ 生成開始... (Total: {total_frames} frames)")
start_time = time.time()

for f in range(total_frames):
    # --- 風の計算 ---
    if WIND_START_F <= f < WIND_PEAK_F:
        t = (f - WIND_START_F) / (WIND_PEAK_F - WIND_START_F)
        current_wind_force = WIND_MAX_STRENGTH * WIND_DIRECTION * math.sin(t * math.pi / 2)
    elif WIND_PEAK_F <= f < WIND_END_F:
        t = (f - WIND_PEAK_F) / (WIND_PEAK_F - WIND_END_F)
        current_wind_force = WIND_MAX_STRENGTH * WIND_DIRECTION * math.cos(t * math.pi / 2)
    else:
        current_wind_force = 0.0

    # --- 爆発の実行 ---
    if f in EXPLOSION_FRAMES:
        if f == 0:
            explosion_y = int(ROWS * PREFILL_HEIGHT * 0.5)
            velocity_grid, (ex, ey) = apply_deep_explosion(grid, velocity_grid)
            ey = explosion_y
        else:
            velocity_grid, (ex, ey) = apply_deep_explosion(grid, velocity_grid)

    # --- 砂の供給 ---
    if f < EMIT_STOP_F:
        for _ in range(3):
            sx = np.random.randint(COLS // 2 - 10, COLS // 2 + 11)
            if not np.any(grid[0, sx]):
                grid[0, sx] = get_dynamic_color()

    # 物理更新
    velocity_grid = apply_wind(grid, velocity_grid, current_wind_force)
    grid, velocity_grid = update_physics_vectorized(grid, velocity_grid, MODE)

    # --- レンダリング (砂のみ回転) ---
    sand_layer = cv2.resize(grid, (WIDTH, HEIGHT), interpolation=cv2.INTER_NEAREST)

    if f >= ROTATION_START_F:
        p = (f - ROTATION_START_F) / (total_frames - ROTATION_START_F)
        p_smooth = p * p * (3 - 2 * p)
        angle = p_smooth * 180
        center = (WIDTH // 2, HEIGHT // 2)
        M = cv2.getRotationMatrix2D(center, angle, 1.0)
        sand_layer = cv2.warpAffine(sand_layer, M, (WIDTH, HEIGHT), flags=cv2.INTER_NEAREST)

    frame = bg_img.copy()
    mask = np.any(sand_layer > 0, axis=2)
    frame[mask] = sand_layer[mask]

    # --- カウントダウンテキスト（黒縁取り付き） ---
    if f < ROTATION_START_F:
        next_event_f_list = [exp_f for exp_f in EXPLOSION_FRAMES if exp_f > f]
        if next_event_f_list:
            remaining = (min(next_event_f_list) - f) / FPS
            text = f"NEXT BOOM: {remaining:.1f}s"
            pos = (60, 120)
            font = cv2.FONT_HERSHEY_SIMPLEX
            scale = 1.4
            cv2.putText(frame, text, pos, font, scale, (0, 0, 0), 10, cv2.LINE_AA)
            cv2.putText(frame, text, pos, font, scale, (255, 255, 255), 3, cv2.LINE_AA)

    frames_out.append(frame)

    # --- 進捗表示（2%刻み） ---
    if f % max(1, (total_frames // 50)) == 0:
        # ここで DURATION に合わせた進捗が表示される
        print(f"Progress: {f*100//total_frames}%")

print(f"\n✅ 完了！ ({DURATION}s) 処理時間: {time.time() - start_time:.2f}秒")

🚀 20秒ループ 生成開始... (Total: 500 frames)
Progress: 0%
Progress: 2%
Progress: 4%
Progress: 6%
Progress: 8%
Progress: 10%
Progress: 12%
Progress: 14%
Progress: 16%
Progress: 18%
Progress: 20%
Progress: 22%
Progress: 24%
Progress: 26%
Progress: 28%
Progress: 30%
Progress: 32%
Progress: 34%
Progress: 36%
Progress: 38%
Progress: 40%
Progress: 42%
Progress: 44%
Progress: 46%
Progress: 48%
Progress: 50%
Progress: 52%
Progress: 54%
Progress: 56%
Progress: 58%
Progress: 60%
Progress: 62%
Progress: 64%
Progress: 66%
Progress: 68%
Progress: 70%
Progress: 72%
Progress: 74%
Progress: 76%
Progress: 78%
Progress: 80%
Progress: 82%
Progress: 84%
Progress: 86%
Progress: 88%
Progress: 90%
Progress: 92%
Progress: 94%
Progress: 96%
Progress: 98%

✅ 完了！ (20s) 処理時間: 56.95秒


In [ ]:
import os
import time

# 保存パス（現在の設定を維持）
output_dir = "/content/drive/MyDrive/Colab Notebooks/output"
os.makedirs(output_dir, exist_ok=True)
temp_path = "/content/temp_raw.mp4"

timestamp = time.strftime("%Y%m%d_%H%M%S")
final_path = os.path.join(output_dir, f"sand_art_{DURATION}s_{timestamp}.mp4")

# 1. 一時ファイル生成
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(temp_path, fourcc, FPS, (WIDTH, HEIGHT))

print("📦 映像データを生成中...")
for frame in frames_out:
    out.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
out.release()

# 2. ffmpegで爆速圧縮
# -preset ultrafast を追加して処理時間を大幅短縮
print("⚡ 爆速エンコード中（ultrafast設定）...")
!ffmpeg -y -i "{temp_path}" -vcodec libx264 -crf 23 -preset ultrafast -pix_fmt yuv420p -movflags +faststart "{final_path}" -loglevel error

# 一時ファイルの削除
if os.path.exists(temp_path):
    os.remove(temp_path)

# サイズ確認
file_size_mb = os.path.getsize(final_path) / (1024 * 1024)
print(f"\n✅ 完了！")
print(f"📂 保存先: {final_path}")
print(f"📦 サイズ: {file_size_mb:.2f} MB")

📦 映像データを生成中...
⚡ 爆速エンコード中（ultrafast設定）...

✅ 完了！
📂 保存先: /content/drive/MyDrive/Colab Notebooks/output/sand_art_20s.mp4
📦 サイズ: 16.40 MB
